# Практическая работа №2: Обработка выборочных данных. Нахождение точечных оценок параметров распределения


Выполнили студенты гр. 2383 Миненок Алиса и Мордасов Евгений. Вариант №27.

## Цель работы

Получение практических навыков нахождения точечных статистических оценок параметров распределения.



## Основные теоретические положения

**Точечная статистическая оценка** — это оценка параметра генеральной совокупности, определяемая одним числом, вычисленным на основе выборочных данных. В данной работе исследуются оценки математического ожидания, дисперсии, а также показатели формы распределения и структурные средние.

#### 1. Основные числовые характеристики
Для несгруппированной выборки объема $n$ вычисляются:
* **Выборочное среднее**: $\bar{x}_в = \frac{1}{n}\sum_{i=1}^{n} x_i$
* **Смещенная выборочная дисперсия**: $D_в = \frac{1}{n}\sum_{i=1}^{n} (x_i - \bar{x}_в)^2$. Является смещенной оценкой.
* **Исправленная выборочная дисперсия**: $s^2 = \frac{n}{n-1}D_в$. Несмещенная оценка дисперсии генеральной совокупности.
* **Исправленное среднее квадратическое отклонение (СКО)**: $s = \sqrt{s^2}$.

#### 2. Метод моментов и условные варианты
Для упрощения вычислений по интервальному ряду применяют переход к условным вариантам:
$$u_i = \frac{x_i - C}{h}$$
где $x_i$ — середина $i$-го интервала, $h$ — шаг интервала, $C$ (условный нуль) — середина интервала с наибольшей частотой.

* **Условные эмпирические моменты** $k$-го порядка: $\overline{M}_k^* = \frac{1}{n} \sum n_i u_i^k$
* **Центральные эмпирические моменты** $\overline{m}_k$ вычисляются через условные:
  * $\overline{m}_1 = 0$
  * $\overline{m}_2 = (\overline{M}_2^* - (\overline{M}_1^*)^2) \cdot h^2 = D_{в(усл)}$
  * $\overline{m}_3 = (\overline{M}_3^* - 3\overline{M}_2^*\overline{M}_1^* + 2(\overline{M}_1^*)^3) \cdot h^3$
  * $\overline{m}_4 = (\overline{M}_4^* - 4\overline{M}_3^*\overline{M}_1^* + 6\overline{M}_2^*(\overline{M}_1^*)^2 - 3(\overline{M}_1^*)^4) \cdot h^4$

#### 3. Показатели формы распределения
* **Коэффициент асимметрии** $\overline{A}_s = \frac{\overline{m}_3}{s^3}$. Характеризует скошенность распределения относительно центра (при $\overline{A}_s = 0$ распределение симметрично).
* **Эксцесс** $\overline{E} = \frac{\overline{m}_4}{s^4} - 3$. Характеризует островершинность или плосковершинность (платикуртичность) распределения по сравнению с нормальным законом.

#### 4. Структурные характеристики и коэффициент вариации
* **Мода ($M_o^*$)** — наиболее часто встречающееся значение признака (определяется по модальному интервалу).
* **Медиана ($M_e^*$)** — значение признака, делящее ранжированную выборку на две равные части.
* **Коэффициент вариации** $V^* = \frac{s}{|\bar{x}_в|} \cdot 100\%$. Характеризует относительную меру разброса и однородность выборки (если $V^* \le 33\%$, выборка считается однородной).

## Постановка задачи

Для заданных выборочных данных вычислить с использованием метода моментов и условных вариант точечные статистические оценки математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации исследуемой случайной величины. Полученные результаты содержательно проинтерпретировать.

## Выполнение работы




### Пункт 1. Вычисление условных вариант.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Загрузка выборки
df = pd.read_csv('sample_27.csv')
X = df['X'].values
n = len(X)

# Формирование интервального ряда
X_sorted = np.sort(X)
k = int(np.round(1 + 3.322 * np.log10(n)))
X_min, X_max = X_sorted.min(), X_sorted.max()
h = (X_max - X_min) / k

bins = [X_min + i * h for i in range(k+1)]
bins[-1] += 1e-5

intervals = pd.cut(X, bins=bins, right=False)
n_i = intervals.value_counts().sort_index().values # абсолютные частоты
x_i = [(bins[i] + bins[i+1])/2 for i in range(k)]  # середины интервалов

print(f"Объем выборки n = {n}, количество интервалов k = {k}, шаг h = {h:.4f}")

Объем выборки n = 120, количество интервалов k = 8, шаг h = 4.0562


Переходим от середин интервалов $x_i$ к условным вариантам $u_i = \frac{x_i - C}{h}$ для упрощения вычислений. В качестве условного нуля $C$ выбираем середину интервала с наибольшей частотой. Вычисляем необходимые произведения для метода моментов и проводим контрольную проверку сумм.

Сформируем таблицу 1 согласно заданию.

In [10]:
# Находим ложный нуль C
max_freq_index = np.argmax(n_i)
C = x_i[max_freq_index]

# Условные варианты u_i
u_i = np.round((np.array(x_i) - C) / h, 3)

# Расчет столбцов Таблицы 1
n_i_u_i = n_i * u_i
n_i_u_i_2 = n_i * (u_i ** 2)
n_i_u_i_3 = n_i * (u_i ** 3)
n_i_u_i_4 = n_i * (u_i ** 4)
n_i_u_i_plus_1_4 = n_i * ((u_i + 1) ** 4)

# Формируем Таблицу 1
table1 = pd.DataFrame({'i': range(1, k+1), 'x_i': np.round(x_i, 3), 'n_i': n_i, 'u_i': u_i,
                       'n_i*u_i': np.round(n_i_u_i, 4), 'n_i*u_i^2': np.round(n_i_u_i_2, 4),
                       'n_i*u_i^3': np.round(n_i_u_i_3, 4), 'n_i*u_i^4': np.round(n_i_u_i_4, 4),
                       'n_i*(u_i+1)^4': np.round(n_i_u_i_plus_1_4, 4)})

# Σ
sum_row = pd.DataFrame({'i': ['Σ'], 'x_i': ['-'], 'n_i': [np.sum(n_i)], 'u_i': ['-'],
                        'n_i*u_i': [np.round(np.sum(n_i_u_i), 4)], 'n_i*u_i^2': [np.round(np.sum(n_i_u_i_2), 4)],
                        'n_i*u_i^3': [np.round(np.sum(n_i_u_i_3), 4)], 'n_i*u_i^4': [np.round(np.sum(n_i_u_i_4), 4)],
                        'n_i*(u_i+1)^4': [np.round(np.sum(n_i_u_i_plus_1_4), 4)]})

display(pd.concat([table1, sum_row], ignore_index=True))

sum_left = np.sum(n_i_u_i_plus_1_4)
sum_right = np.sum(n_i_u_i_4) + 4*np.sum(n_i_u_i_3) + 6*np.sum(n_i_u_i_2) + 4*np.sum(n_i_u_i) + np.sum(n_i)

is_correct = np.isclose(sum_left, sum_right, atol=0.05)

print(f"Контрольная проверка: {sum_left:.4f} ≈ {sum_right:.4f} -> {'ВЕРНО' if is_correct else 'ОШИБКА'}")

,i,x_i,n_i,u_i,n_i*u_i,n_i*u_i^2,n_i*u_i^3,n_i*u_i^4,n_i*(u_i+1)^4
0,1,5.378,15,-7.0,-105.0,735.0,-5145.0,36015.0,19440.0
1,2,9.434,17,-6.0,-102.0,612.0,-3672.0,22032.0,10625.0
2,3,13.491,14,-5.0,-70.0,350.0,-1750.0,8750.0,3584.0
3,4,17.547,14,-4.0,-56.0,224.0,-896.0,3584.0,1134.0
4,5,21.603,17,-3.0,-51.0,153.0,-459.0,1377.0,272.0
5,6,25.659,4,-2.0,-8.0,16.0,-32.0,64.0,4.0
6,7,29.716,13,-1.0,-13.0,13.0,-13.0,13.0,0.0
7,8,33.772,26,0.0,0.0,0.0,0.0,0.0,26.0
8,Σ,-,120,-,-405.0,2103.0,-11967.0,71835.0,35085.0


Контрольная проверка: 35085.0000 ≈ 35085.0000 -> ВЕРНО


### Пункт 2. Вычисление эмпирических моментов.
Вычисляем условные эмпирические моменты $\overline{M}_k^* = \frac{1}{n} \sum n_i u_i^k$. С их помощью по формулам связи находим центральные эмпирические моменты $\overline{m}_k$, которые понадобятся для дисперсии, асимметрии и эксцесса.


Сформируем таблицу 2 согласно заданию.

In [11]:
# Условные моменты M_k^*
M1_star = np.sum(n_i_u_i) / n
M2_star = np.sum(n_i_u_i_2) / n
M3_star = np.sum(n_i_u_i_3) / n
M4_star = np.sum(n_i_u_i_4) / n

# Центральные моменты m_k
m1 = 0
m2 = (M2_star - M1_star**2) * (h**2)
m3 = (M3_star - 3*M2_star*M1_star + 2*(M1_star**3)) * (h**3)
m4 = (M4_star - 4*M3_star*M1_star + 6*M2_star*(M1_star**2) - 3*(M1_star**4)) * (h**4)

table2 = pd.DataFrame({'k': [1, 2, 3, 4],
                       'bar{M}_k^*': [M1_star, M2_star, M3_star, M4_star],
                       'bar{m}_k': [m1, m2, m3, m4]})
display(table2.round(4))

,k,bar{M}_k^*,bar{m}_k
0,1,-3.375,0.0000
1,2,17.525,100.9299
2,3,-99.725,55.3197
3,4,598.625,16465.1082


### Пункт 3. Выборочные среднее, дисперсия и СКО.

Рассчитываем выборочное среднее $\bar{x}_в$ и смещенную дисперсию $D_в$ двумя способами (по стандартной формуле для несгруппированных данных и через условные варианты). Убеждаемся в совпадении результатов и находим выборочное СКО $\sigma_в$.

In [12]:
# По условным вариантам - сгруппированные данные
mean_u = M1_star * h + C
var_u = m2

# По стандартной формуле - несгруппированные данные
mean_std = np.mean(X)
var_std = np.var(X, ddof=0) # Смещенная дисперсия (D_в)
std_dev_std = np.sqrt(var_std) # Выборочное СКО (σ_в)

print(f"Выборочное среднее (через усл. варианты): {mean_u:.4f}")
print(f"Выборочное среднее (стандартная формула): {mean_std:.4f}")
print(f"Дисперсия D_в (через усл. варианты):      {var_u:.4f}")
print(f"Дисперсия D_в (стандартная формула):      {var_std:.4f}")
print(f"Выборочное СКО (σ_в):                     {std_dev_std:.4f}")

Выборочное среднее (через усл. варианты): 20.0820
Выборочное среднее (стандартная формула): 20.1412
Дисперсия D_в (через усл. варианты):      100.9299
Дисперсия D_в (стандартная формула):      102.3871
Выборочное СКО (σ_в):                     10.1187


Сравнение результатов показывает, что точечные оценки выборочного среднего ($\bar{x}_в$) и смещенной дисперсии ($D_в$), вычисленные методом условных вариант (по сгруппированным данным) и по стандартной формуле (по исходной выборке), практически совпадают.

Погрешность объясняется особенностями перехода к интервальному ряду: при группировке мы делаем допущение, что все значения вариант внутри интервала сконцентрированы в его середине ($\tilde{x}_i$).

Тем не менее, метод моментов и условных вариант показал свою высокую эффективность: он позволяет значительно упростить ручные вычисления, получая при этом статистические оценки, крайне близкие к истинным выборочным значениям.

В ходе выполнения пункта исходная непрерывная выборка объемом $n = 120$ была преобразована в ранжированный, а затем в интервальный статистический ряд. Поскольку генеральная совокупность представлена непрерывными значениями, построение классического вариационного ряда нецелесообразно. Группировка данных в интервалы позволила сгладить шум индивидуальных значений и выявить закономерности распределения.

Сформированная Таблица 1 подтверждает математическую корректность группировки:
* Итоговая сумма столбца абсолютных частот строго совпадает с общим объемом выборки ($\sum m_i = 120$).
* Сумма относительных частот равна единице ($\sum \tilde{m}_i = 1.0$).
* Последнее значение накопленной абсолютной частоты ($m_k^{нак}$) достигает $120$, что доказывает отсутствие потерь данных при разбиении.

### Пункт 4. Вычисление исправленной дисперсии и СКО

Рассчитываем несмещенные оценки дисперсии $s^2 = \frac{n}{n-1}D_в$ и исправленное СКО $s$. Сравниваем их со смещенными оценками.

In [13]:
var_corrected = (n / (n - 1)) * var_std # Исправленная дисперсия s^2
std_dev_corrected = np.sqrt(var_corrected) # Исправленное СКО s

print(f"Исправленная дисперсия (s^2): {var_corrected:.4f}")
print(f"Исправленное СКО (s):         {std_dev_corrected:.4f}")

Исправленная дисперсия (s^2): 103.2475
Исправленное СКО (s):         10.1611


Стандартная выборочная дисперсия $D_в$ является смещенной, то есть занижает истинную дисперсию генеральной совокупности. Применение поправочного множителя $\frac{n}{n-1}$ (поправка Бесселя) позволяет получить исправленную несмещенную оценку $s^2$. Так как объе выборки $n=120$, разница между смещенной и несмещенной оценками незначительна (порядка 1%). Однако с математической точки зрения для дальнейших расчетов следует использовать именно исправленное СКО $s$, как более точную оценку параметра генеральной совокупности.

### Пункт 5. Вычисление коэффициентов асимметрии и эксцесса

Оцениваем форму распределения. Вычисляем коэффициент асимметрии $\overline{A}_s = \frac{\overline{m}_3}{s^3}$ и эксцесс $\overline{E} = \frac{\overline{m}_4}{s^4} - 3$.

In [14]:
As = m3 / (std_dev_corrected ** 3)
Ek = (m4 / (std_dev_corrected ** 4)) - 3

print(f"Коэффициент асимметрии (A_s) = {As:.4f}")
print(f"Эксцесс (E)                  = {Ek:.4f}")

Коэффициент асимметрии (A_s) = 0.0527
Эксцесс (E)                  = -1.4554


* Рассчитанный коэффициент асимметрии $\overline{A}_s \approx 0.05$. Поскольку значение очень близко к нулю, это указывает на то, что эмпирическое распределение практически симметрично с едва заметной правосторонней скошенностью.
* Рассчитанный коэффициент эксцесса $\overline{E} \approx -1.45$. Это означает, что распределение является плосковершинное. По сравнению с эталонным нормальным распределением (где $E=0$), наша выборка имеет более "легкие" хвосты и плоскую, "размазанную" вершину в центральной части.

### Пункт 6. Вычисление моды, медианы и коэффициента вариации

Находим структурные средние: моду $M_o^*$ и медиану $M_e^*$.
Вычисляем коэффициент вариации $V^* = \frac{s}{|\bar{x}_в|} \cdot 100\%$ для оценки однородности выборки.

In [15]:
# Мода
mod_idx = np.argmax(n_i)
x_Mo_min = bins[mod_idx]
n_Mo = n_i[mod_idx]
n_Mo_prev = n_i[mod_idx - 1] if mod_idx > 0 else 0
n_Mo_next = n_i[mod_idx + 1] if mod_idx < (k - 1) else 0
Mo = x_Mo_min + h * ((n_Mo - n_Mo_prev) / ((n_Mo - n_Mo_prev) + (n_Mo - n_Mo_next)))

# Медиана
cum_n = np.cumsum(n_i)
med_idx = np.where(cum_n >= n / 2)[0][0]
x_Me_min = bins[med_idx]
cum_n_prev = cum_n[med_idx - 1] if med_idx > 0 else 0
n_Me = n_i[med_idx]
Me = x_Me_min + h * ((n/2 - cum_n_prev) / n_Me)

# Коэффициент вариации
V = (std_dev_corrected / abs(mean_std)) * 100

print(f"Мода (M_o^*)                 = {Mo:.4f}")
print(f"Медиана (M_e^*)              = {Me:.4f}")
print(f"Коэффициент вариации (V^*)   = {V:.2f}%")

Мода (M_o^*)                 = 33.0958
Медиана (M_e^*)              = 19.5750
Коэффициент вариации (V^*)   = 50.45%


* Структурные характеристики дают дополнительное представление о центре распределения.
* Вычисленный коэффициент вариации $V^* \approx 50.45\%$. В статистике принято считать, что если $V^* > 33\%$, то выборка является статистически неоднородной. Большой разброс значений относительно выборочного среднего указывает на то, что исследуемый признак варьируется в широком диапазоне, и среднее арифметическое не является строго типичным представителем для всех элементов данной совокупности.

### Вывод


В ходе практической работы было проведено нахождение точечных статистических оценок параметров распределения для одномерной непрерывной выборки. Были выполнены процедуры вычисления выборочных моментов, рассчитаны оценки математического ожидания, дисперсии и среднеквадратичного отклонения как по стандартным формулам, так и с использованием метода моментов и условных вариант. Дополнительно были вычислены показатели формы распределения (асимметрия, эксцесс) и структурные средние (мода, медиана, коэффициент вариации).

На основе анализа полученных результатов установлено, что:

* Применение метода моментов с переходом к условным вариантам (использованием условного нуля) позволяет существенно упростить вычислительные процедуры при работе со сгруппированными данными, обеспечивая при этом высокую точность точечных оценок, сопоставимую с расчетами по несгруппированной выборке.
* Использование поправки Бесселя (переход от смещенной дисперсии к исправленной) компенсирует систематическое занижение разброса данных, что является необходимым шагом для получения несмещенных точечных оценок параметров генеральной совокупности.
* Анализ коэффициентов асимметрии и эксцесса позволил количественно оценить форму эмпирического распределения, выявив его практическую симметричность и плосковершинность по сравнению с эталонным нормальным законом.
* Вычисление коэффициента вариации (порядка 50%) показало высокую степень рассеяния значений относительно центра распределения, что свидетельствует о статистической неоднородности исследуемой выборки.

Таким образом, в результате выполнения работы были приобретены практические навыки вычисления точечных статистических оценок параметров распределения, освоен алгоритм применения метода моментов через условные варианты, а также сформировано глубокое понимание содержательного смысла числовых и структурных характеристик случайной величины.